# Stream A — Improved Pipeline
## Whisper → Pyannote → OpenSMILE (eGeMAPSv02)

| Step | Tool | Purpose |
|------|------|---------|
| 0 | Config | Paths, tokens, parameters |
| 1 | Environment check | Verify all dependencies |
| 2 | Subject discovery | Scan data directory |
| 3 | Whisper | Transcription + timestamps |
| 4 | Pyannote | Speaker diarization — isolate participant only |
| 5 | Segment export | Export participant-only WAV segments |
| 6 | OpenSMILE | Extract eGeMAPSv02 (88 features) per segment |
| 7 | Aggregation | mean/std/p10/p50/p90 → one row per subject |
| 8 | Merge labels | Join PHQ-8 scores from train/dev split CSV |
| 9 | Output + QC | Save CSV, validate, plot feature distributions |

> **Key improvement over v1**: Pyannote filters out Ellie (interviewer) before feature extraction,
> so features reflect participant speech only. eGeMAPSv02 is the academic standard for
> depression/emotion recognition — directly comparable to published results.

---
## Cell 0 — Configuration

In [ ]:
from pathlib import Path

# ============================================================
# ✏️  Edit this section only
# ============================================================

# --- Paths ---
RAW_DATA_DIR  = Path("/Users/Meihui/Downloads/sync/work/whisperx")
LABELS_DIR    = RAW_DATA_DIR  # train_split / dev_split CSVs should be here or adjust
OUTPUT_DIR    = Path("outputs/features")
SEGMENT_DIR   = Path("outputs/segments")
LOG_DIR       = Path("outputs/logs")

# --- HuggingFace token (needed for Pyannote) ---
import os
HF_TOKEN = os.getenv("HF_TOKEN")   # ✏️ Replace with your actual token

# --- Model settings ---
WHISPER_MODEL_SIZE = "base"        # tiny / base / small / medium
PYANNOTE_MODEL    = "pyannote/speaker-diarization-3.1"
PARTICIPANT_LABEL = "PARTICIPANT"  # how participant is labelled in DAIC-WOZ transcripts

# --- Audio settings ---
TARGET_SR           = 16000
MIN_SEGMENT_DUR_SEC = 0.5   # longer minimum for eGeMAPSv02 stability

# --- Output ---
OUTPUT_CSV = OUTPUT_DIR / "improved_features.csv"

# ============================================================

for d in [OUTPUT_DIR, SEGMENT_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Configuration loaded")
print(f"  Data dir   : {RAW_DATA_DIR}")
print(f"  HF token   : {HF_TOKEN[:8]}...")
print(f"  Output CSV : {OUTPUT_CSV}")

---
## Cell 1 — Environment Check

In [ ]:
import sys, json, time, logging, warnings
from datetime import datetime

import numpy as np
import pandas as pd
import soundfile as sf
import librosa
import torch
import opensmile
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

# --- Version report ---
print(f"Python      : {sys.version.split()[0]}")
print(f"PyTorch     : {torch.__version__}")
print(f"OpenSMILE   : {opensmile.__version__}")
print(f"NumPy       : {np.__version__}")
print(f"Pandas      : {pd.__version__}")

# --- Device ---
DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"\nDevice      : {DEVICE}")

# --- Verify OpenSMILE eGeMAPSv02 is available ---
smile = opensmile.Smile(
    feature_set=opensmile.FeatureSet.eGeMAPSv02,
    feature_level=opensmile.FeatureLevel.Functionals,
)
print(f"eGeMAPSv02 features : {len(smile.feature_names)} features")
print(f"First 5             : {smile.feature_names[:5]}")

# --- Logger ---
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler(LOG_DIR / f"run_{RUN_ID}.log"),
        logging.StreamHandler(sys.stdout),
    ],
)
logger = logging.getLogger("improved")
logger.info(f"Pipeline started | run_id={RUN_ID} | device={DEVICE}")
print("\n✅ Environment OK")

---
## Cell 2 — Subject Discovery

In [ ]:
def discover_subjects(raw_dir: Path) -> list:
    """
    Scan raw_dir for subject folders.
    Expects: raw_dir/{subject_id}/{subject_id}_AUDIO.wav
    e.g.   : whisperx/300_P/300_AUDIO.wav
    """
    subjects = []
    if not raw_dir.exists():
        print(f"ERROR: Directory not found: {raw_dir}")
        return subjects

    for d in sorted(raw_dir.iterdir()):
        if not d.is_dir():
            continue
        # DAIC-WOZ naming: folder=300_P, audio=300_AUDIO.wav
        session_id = d.name.replace("_P", "")  # 300_P -> 300
        audio = d / f"{session_id}_AUDIO.wav"

        # Fallback: auto-detect first WAV
        if not audio.exists():
            wavs = sorted(d.glob("*.wav"))
            if not wavs:
                print(f"  SKIP {d.name}: no WAV found")
                continue
            audio = wavs[0]

        # Check for official transcript
        transcript = d / f"{session_id}_TRANSCRIPT.csv"

        subjects.append({
            "subject_id" : d.name,
            "session_id" : session_id,
            "audio_path" : audio,
            "transcript" : transcript if transcript.exists() else None,
        })
        t_status = "transcript ✅" if transcript.exists() else "transcript ❌"
        print(f"  [{d.name}]  {audio.name}  |  {t_status}")

    return subjects


SUBJECTS = discover_subjects(RAW_DATA_DIR)
print(f"\nTotal subjects: {len(SUBJECTS)}")

n_with_transcript = sum(1 for s in SUBJECTS if s["transcript"] is not None)
print(f"With official transcript: {n_with_transcript}/{len(SUBJECTS)}")

---
## Cell 3 — Whisper Transcription

Used as a fallback for subjects without official transcripts.
For subjects WITH `XXX_TRANSCRIPT.csv`, we use the official one (more accurate).

In [ ]:
def load_official_transcript(transcript_path: Path) -> list:
    """
    Load DAIC-WOZ official transcript.
    Format: start_time \t stop_time \t speaker \t value
    Returns participant-only segments as {start, end, text, speaker}
    """
    try:
        df = pd.read_csv(transcript_path, sep="\t",
                         names=["start", "end", "speaker", "text"])
        # Keep participant utterances only (filter out Ellie)
        participant = df[df["speaker"] == "Participant"].copy()
        # Also filter scrubbed entries
        participant = participant[~participant["text"].str.contains(
            "scrubbed_entry", na=False)]
        segments = [
            {"start": row.start, "end": row.end,
             "text": str(row.text), "speaker": "PARTICIPANT",
             "source": "official_transcript"}
            for _, row in participant.iterrows()
        ]
        return segments
    except Exception as e:
        logger.warning(f"Failed to load transcript {transcript_path}: {e}")
        return []


def transcribe_whisper(audio_path: Path, model_size: str) -> list:
    """Fallback: transcribe with faster-whisper."""
    try:
        from faster_whisper import WhisperModel
        model = WhisperModel(model_size, device="cpu", compute_type="int8")
        segs, _ = model.transcribe(
            str(audio_path), beam_size=5, language="en", vad_filter=True
        )
        return [{"start": s.start, "end": s.end, "text": s.text.strip(),
                 "speaker": "UNKNOWN", "source": "whisper"}
                for s in segs]
    except ImportError:
        import whisper
        model = whisper.load_model(model_size)
        result = model.transcribe(str(audio_path), language="en")
        return [{"start": s["start"], "end": s["end"],
                 "text": s["text"].strip(), "speaker": "UNKNOWN",
                 "source": "whisper"}
                for s in result["segments"]]


# --- Run ---
ALL_TRANSCRIPTS = {}

for subj in tqdm(SUBJECTS, desc="Transcription"):
    sid = subj["subject_id"]
    cache = LOG_DIR / f"{sid}_transcript.json"

    if cache.exists():
        with open(cache) as f:
            segments = json.load(f)
        logger.info(f"{sid}: loaded from cache ({len(segments)} segments)")
    elif subj["transcript"] is not None:
        # Use official DAIC-WOZ transcript (already filtered to participant)
        segments = load_official_transcript(subj["transcript"])
        logger.info(f"{sid}: official transcript ({len(segments)} participant segments)")
        with open(cache, "w") as f:
            json.dump(segments, f, indent=2)
    else:
        # Fallback to Whisper
        segments = transcribe_whisper(subj["audio_path"], WHISPER_MODEL_SIZE)
        logger.info(f"{sid}: whisper transcription ({len(segments)} segments)")
        with open(cache, "w") as f:
            json.dump(segments, f, indent=2)

    ALL_TRANSCRIPTS[sid] = segments

print("\nTranscription summary:")
for sid, segs in ALL_TRANSCRIPTS.items():
    sources = set(s.get("source", "?") for s in segs)
    print(f"  [{sid}]  {len(segs)} segments  |  source: {sources}")

---
## Cell 4 — Pyannote Speaker Diarization

Identifies which speaker is the participant vs Ellie.
For subjects with official transcripts this is a cross-check.
For subjects without transcripts, this is the primary filter.

In [ ]:
# from pyannote.audio import Pipeline as PyannotePipeline

# def load_pyannote(hf_token: str, model_name: str):
#     """Load pyannote diarization pipeline."""
#     logger.info(f"Loading Pyannote: {model_name}")
#     pipeline = PyannotePipeline.from_pretrained(
#         model_name,
#         use_auth_token=hf_token
#     )
#     # Run on CPU to avoid MPS instability
#     pipeline.to(torch.device("cpu"))
#     return pipeline


# def diarize_audio(pipeline, audio_path: Path) -> list:
#     """
#     Run speaker diarization on audio.
#     Returns list of {start, end, speaker_label}
#     """
#     diarization = pipeline(str(audio_path))
#     segments = []
#     for turn, _, speaker in diarization.itertracks(yield_label=True):
#         segments.append({
#             "start"  : turn.start,
#             "end"    : turn.end,
#             "speaker": speaker,   # e.g. SPEAKER_00, SPEAKER_01
#         })
#     return segments


# def identify_participant_speaker(diarization_segments: list,
#                                   transcript_segments: list) -> str:
#     """
#     Cross-reference diarization output with transcript to identify
#     which Pyannote speaker label corresponds to the participant.

#     Strategy: participant usually speaks more total time than Ellie.
#     Returns the speaker label with the most total speaking time.
#     """
#     speaker_durations = {}
#     for seg in diarization_segments:
#         spk = seg["speaker"]
#         dur = seg["end"] - seg["start"]
#         speaker_durations[spk] = speaker_durations.get(spk, 0) + dur

#     if not speaker_durations:
#         return None

#     # Participant typically speaks more than the interviewer
#     participant_label = max(speaker_durations, key=speaker_durations.get)
#     logger.info(f"Speaker durations: {speaker_durations}")
#     logger.info(f"Identified participant as: {participant_label}")
#     return participant_label


# def filter_participant_segments(diarization_segments: list,
#                                  participant_label: str,
#                                  min_dur: float = MIN_SEGMENT_DUR_SEC) -> list:
#     """Return only segments belonging to the participant."""
#     return [
#         seg for seg in diarization_segments
#         if seg["speaker"] == participant_label
#         and (seg["end"] - seg["start"]) >= min_dur
#     ]


# # --- Run ---
# logger.info("Loading Pyannote pipeline...")
# pyannote_pipeline = load_pyannote(HF_TOKEN, PYANNOTE_MODEL)
# logger.info("Pyannote loaded")

# ALL_PARTICIPANT_SEGMENTS = {}

# for subj in tqdm(SUBJECTS, desc="Diarization"):
#     sid = subj["subject_id"]
#     cache = LOG_DIR / f"{sid}_diarization.json"

#     if cache.exists():
#         with open(cache) as f:
#             participant_segs = json.load(f)
#         logger.info(f"{sid}: diarization loaded from cache ({len(participant_segs)} segs)")
#     else:
#         try:
#             diar_segs = diarize_audio(pyannote_pipeline, subj["audio_path"])
#             participant_label = identify_participant_speaker(
#                 diar_segs, ALL_TRANSCRIPTS.get(sid, [])
#             )
#             participant_segs = filter_participant_segments(diar_segs, participant_label)
#             with open(cache, "w") as f:
#                 json.dump(participant_segs, f, indent=2)
#             logger.info(f"{sid}: diarized | {len(participant_segs)} participant segments")
#         except Exception as e:
#             logger.error(f"{sid}: diarization failed — {e}")
#             # Fallback: use transcript segments as-is
#             participant_segs = ALL_TRANSCRIPTS.get(sid, [])
#             logger.info(f"{sid}: falling back to transcript segments")

#     ALL_PARTICIPANT_SEGMENTS[sid] = participant_segs

# print("\nDiarization summary:")
# for sid, segs in ALL_PARTICIPANT_SEGMENTS.items():
#     total = sum(s["end"] - s["start"] for s in segs)
#     print(f"  [{sid}]  {len(segs)} participant segments  |  {total:.1f}s speech")

In [ ]:
# Skip Pyannote — official transcripts already identify participant segments
ALL_PARTICIPANT_SEGMENTS = ALL_TRANSCRIPTS.copy()
print("Using official transcript segments directly — Pyannote skipped")

---
## Cell 5 — Export Participant WAV Segments

In [ ]:
def export_segments(audio_path: Path, segments: list,
                    out_dir: Path, target_sr: int = TARGET_SR) -> list:
    """
    Slice audio into per-segment WAV files.
    Returns segments list with wav_path added.
    """
    out_dir.mkdir(parents=True, exist_ok=True)
    audio, _ = librosa.load(str(audio_path), sr=target_sr, mono=True)

    # Peak normalize to remove microphone gain differences
    peak = np.max(np.abs(audio))
    if peak > 1e-6:
        audio = audio * (0.95 / peak)

    valid = []
    for i, seg in enumerate(segments):
        start = float(seg.get("start", 0))
        end   = float(seg.get("end", 0))
        dur   = end - start
        if dur < MIN_SEGMENT_DUR_SEC:
            continue

        s = int(start * target_sr)
        e = min(int(end * target_sr), len(audio))
        chunk = audio[s:e]

        if len(chunk) < int(MIN_SEGMENT_DUR_SEC * target_sr):
            continue

        wav_path = out_dir / f"seg_{i:04d}.wav"
        sf.write(str(wav_path), chunk, target_sr)
        valid.append({**seg, "wav_path": str(wav_path), "duration": dur})

    return valid


ALL_EXPORTED_SEGMENTS = {}

for subj in tqdm(SUBJECTS, desc="Exporting segments"):
    sid  = subj["subject_id"]
    segs = ALL_PARTICIPANT_SEGMENTS.get(sid, [])

    exported = export_segments(
        subj["audio_path"], segs, SEGMENT_DIR / sid
    )
    ALL_EXPORTED_SEGMENTS[sid] = exported
    logger.info(f"{sid}: exported {len(exported)} segments")

print("\nExport summary:")
for sid, segs in ALL_EXPORTED_SEGMENTS.items():
    total = sum(s["duration"] for s in segs)
    print(f"  [{sid}]  {len(segs)} segments  |  {total:.1f}s  |  "
          f"avg {np.mean([s['duration'] for s in segs]):.1f}s" if segs else f"  [{sid}]  0 segments")

---
## Cell 6 — OpenSMILE Feature Extraction (eGeMAPSv02)

eGeMAPSv02 = 88 features covering F0, loudness, spectral, MFCC, voice quality.
Academic standard for depression/emotion recognition research.

In [ ]:
# Initialise OpenSMILE with eGeMAPSv02
smile = opensmile.Smile(
    feature_set=opensmile.FeatureSet.eGeMAPSv02,
    feature_level=opensmile.FeatureLevel.Functionals,  # summary stats per segment
)
print(f"eGeMAPSv02: {len(smile.feature_names)} features")
print("Feature list:")
for i, name in enumerate(smile.feature_names):
    print(f"  {i+1:3d}. {name}")


def extract_egemaps(wav_path: str) -> dict:
    """
    Extract eGeMAPSv02 features from a single WAV segment.
    Returns a flat dict of 88 features.
    """
    result = smile.process_file(wav_path)
    # result is a DataFrame with one row
    return result.iloc[0].to_dict()


# --- Run ---
EGEMAPS_FEATURES = {}

for sid, segments in tqdm(ALL_EXPORTED_SEGMENTS.items(), desc="eGeMAPSv02"):
    feats_list = []

    for seg in tqdm(segments, desc=f"  {sid}", leave=False):
        try:
            feats = extract_egemaps(seg["wav_path"])
            feats["seg_start"]    = seg["start"]
            feats["seg_end"]      = seg["end"]
            feats["seg_duration"] = seg["duration"]
            feats_list.append(feats)
        except Exception as e:
            logger.warning(f"{sid} seg {Path(seg['wav_path']).name}: {e}")

    EGEMAPS_FEATURES[sid] = feats_list
    logger.info(f"{sid}: eGeMAPSv02 extracted for {len(feats_list)} segments")

print("\nFeature extraction summary:")
for sid, feats in EGEMAPS_FEATURES.items():
    n_feats = len(feats[0]) if feats else 0
    print(f"  [{sid}]  {len(feats)} segments  |  {n_feats} features/segment")

---
## Cell 7 — Subject-level Aggregation

In [ ]:
PERCENTILES = [10, 50, 90]

def aggregate(sid: str, feats: list) -> dict:
    """
    Aggregate segment-level features into a single subject row.
    Statistics: mean, std, p10, p50, p90 per feature.
    """
    row = {"subject_id": sid}
    if not feats:
        return row

    numeric_keys = [
        k for k in feats[0]
        if isinstance(feats[0][k], (int, float))
        and k not in ("seg_start", "seg_end")
    ]
    mat = np.array([[f.get(k, np.nan) for k in numeric_keys] for f in feats],
                   dtype=float)

    for j, key in enumerate(numeric_keys):
        col = mat[:, j]
        col = col[~np.isnan(col)]
        if len(col) == 0:
            for stat in ["mean", "std"] + [f"p{p}" for p in PERCENTILES]:
                row[f"{key}_{stat}"] = np.nan
            continue
        row[f"{key}_mean"] = float(np.mean(col))
        row[f"{key}_std"]  = float(np.std(col))
        for p in PERCENTILES:
            row[f"{key}_p{p}"] = float(np.percentile(col, p))

    row["n_segments"]       = len(feats)
    row["total_speech_sec"] = float(sum(f.get("seg_duration", 0) for f in feats))
    return row


rows = [
    aggregate(s["subject_id"], EGEMAPS_FEATURES.get(s["subject_id"], []))
    for s in SUBJECTS
]
FEATURE_DF = pd.DataFrame(rows)

print(f"Aggregation complete: {FEATURE_DF.shape[0]} subjects x {FEATURE_DF.shape[1]} features")
FEATURE_DF[["subject_id", "n_segments", "total_speech_sec"]]

---
## Cell 8 — Merge PHQ-8 Labels

Joins train/dev split CSVs to add depression labels to the feature matrix.

In [ ]:
def load_labels(labels_dir: Path) -> pd.DataFrame:
    """
    Load and combine train + dev split label CSVs.
    Returns DataFrame with columns: Participant_ID, PHQ8_Binary, PHQ8_Score, Gender
    """
    dfs = []
    for fname in ["train_split_Depression_AVEC2017.csv",
                  "dev_split_Depression_AVEC2017.csv"]:
        fpath = labels_dir / fname
        if fpath.exists():
            df = pd.read_csv(fpath)
            df["split"] = fname.split("_")[0]  # train / dev
            dfs.append(df)
            print(f"  Loaded {fname}: {len(df)} subjects")
        else:
            print(f"  NOT FOUND: {fpath}")

    if not dfs:
        print("  No label files found. Skipping label merge.")
        return pd.DataFrame()

    labels = pd.concat(dfs, ignore_index=True)
    # Normalise participant ID to match subject_id format (e.g. 300 -> 300_P)
    labels["subject_id"] = labels["Participant_ID"].astype(str) + "_P"
    return labels[["subject_id", "PHQ8_Binary", "PHQ8_Score", "Gender", "split"]]


print("Loading labels...")
LABELS_DF = load_labels(LABELS_DIR)

if not LABELS_DF.empty:
    FEATURE_DF = FEATURE_DF.merge(LABELS_DF, on="subject_id", how="left")
    print(f"\nAfter label merge: {FEATURE_DF.shape}")
    print(f"  With PHQ8 label : {FEATURE_DF['PHQ8_Score'].notna().sum()} subjects")
    print(f"  Missing label   : {FEATURE_DF['PHQ8_Score'].isna().sum()} subjects")
    if FEATURE_DF["PHQ8_Binary"].notna().any():
        counts = FEATURE_DF["PHQ8_Binary"].value_counts()
        print(f"  Depressed (>=10): {counts.get(1, 0)}")
        print(f"  Non-depressed   : {counts.get(0, 0)}")
else:
    print("Skipped label merge — label files not found in:", LABELS_DIR)
    print("Place train_split_Depression_AVEC2017.csv in:", LABELS_DIR)

---
## Cell 9 — Output + Validation + QC Plot

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# --- Save CSV ---
FEATURE_DF.to_csv(OUTPUT_CSV, index=False)
print(f"Saved: {OUTPUT_CSV.resolve()}")
print(f"  Shape     : {FEATURE_DF.shape}")
print(f"  File size : {OUTPUT_CSV.stat().st_size / 1e3:.1f} KB")

# --- Validation ---
print("\n" + "=" * 50)
print("Pipeline Validation")
print("=" * 50)

checks = {
    f"CSV exists"                                        : OUTPUT_CSV.exists(),
    f"Subjects processed (actual={FEATURE_DF.shape[0]})": FEATURE_DF.shape[0] > 0,
    f"Feature columns > 100 (actual={FEATURE_DF.shape[1]})" : FEATURE_DF.shape[1] > 100,
    f"No all-NaN rows"                                   : not FEATURE_DF.drop(
        columns=[c for c in ["subject_id","PHQ8_Binary","PHQ8_Score","Gender","split"]
                 if c in FEATURE_DF.columns]).isnull().all(axis=1).any(),
    f"Unique subject per row"                            : FEATURE_DF["subject_id"].nunique() == len(FEATURE_DF),
}

all_passed = True
for criterion, passed in checks.items():
    print(f"  {'PASS' if passed else 'FAIL'}  {criterion}")
    if not passed:
        all_passed = False

# --- QC Plot: F0 + Energy across subjects ---
qc_features = [
    ("F0semitoneFrom27.5Hz_sma3nz_amean_mean",     "F0 Mean (semitone)"),
    ("F0semitoneFrom27.5Hz_sma3nz_stddevNorm_mean", "F0 Std Norm  ← pitch variability"),
    ("loudness_sma3_amean_mean",                    "Loudness Mean"),
    ("loudness_sma3_stddevNorm_mean",               "Loudness Std Norm"),
    ("HNRdBACF_sma3nz_amean_mean",                  "HNR (voice quality)"),
    ("jitterLocal_sma3nz_amean_mean",               "Jitter (voice irregularity)"),
]

available_qc = [(col, label) for col, label in qc_features if col in FEATURE_DF.columns]

if available_qc and len(FEATURE_DF) > 0:
    subjects = FEATURE_DF["subject_id"].tolist()
    x = np.arange(len(subjects))
    n_plots = len(available_qc)

    fig, axes = plt.subplots(n_plots, 1, figsize=(14, n_plots * 2.5))
    if n_plots == 1:
        axes = [axes]

    colors = plt.cm.tab10.colors
    for i, (col, label) in enumerate(available_qc):
        vals = FEATURE_DF[col].values.astype(float)
        group_mean = np.nanmean(vals)
        group_std  = np.nanstd(vals)

        bar_colors = ["tomato" if v > group_mean + 2 * group_std
                                   or v < group_mean - 2 * group_std
                      else colors[i % len(colors)]
                      for v in vals]

        axes[i].bar(x, vals, color=bar_colors, alpha=0.8, edgecolor="white")
        axes[i].axhline(group_mean, color="black", linestyle="--", linewidth=1,
                        label=f"mean={group_mean:.3f}")
        axes[i].axhspan(group_mean - group_std, group_mean + group_std,
                        alpha=0.08, color="black")
        axes[i].set_title(label, fontsize=10, fontweight="bold", loc="left")
        axes[i].set_xticks(x)
        axes[i].set_xticklabels(subjects, rotation=35, ha="right", fontsize=8)
        axes[i].legend(fontsize=8, loc="upper right")

    plt.suptitle("QC: eGeMAPSv02 Key Features Across Subjects\n"
                 "Red bars = >2 SD from group mean",
                 fontsize=12, fontweight="bold")
    plt.tight_layout()

    qc_path = OUTPUT_DIR / "qc_egemaps_features.png"
    plt.savefig(qc_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"\nQC plot saved: {qc_path}")

print("\n" + "=" * 50)
if all_passed:
    print("Pipeline complete.")
    print(f"Output: {OUTPUT_CSV}")
else:
    print("Some checks failed. Review logs above.")